# Tree-type networks, part 2: augmentation, ensembles, bias-variance, loss landscape
**KAGGLE version.** Settings (right panel): Accelerator = GPU T4 x2 (use one), Internet = On, Persistence = Files only. Everything is saved to `/kaggle/working/results_aug`; if the session stops, results stay in the Output tab and every script **resumes** if you re-run.

| step | what | rough time on a T4 |
|---|---|---|
| 0 | setup, unit tests, **quick trial** (tiny versions of every script) | ~10 min (estimate, untested) |
| 1 | double descent with augmentation (`--augment`), compared with your earlier no-augmentation run | ~45 min (per-width times measured in your earlier run: 160-590 s, 10 widths) |
| 2 | single tree on A augmented views, + fixed-width sweep | unmeasured: tens of minutes |
| 3 | ensemble of 8 trees (majority vote) + ablations | unmeasured: ~1 h |
| 4 | bias-variance (trees vs n; ResNets vs epochs) | unmeasured: ~1-1.5 h |
| 5 | loss landscape | ~15-30 min |

"Unmeasured" means I could not run the torch code where I wrote it; run the quick trial first and read the
printed seconds before committing to the long runs. Each long cell has cheaper settings in a comment.
If you are short of compute units, run steps 2, 3 and 1 in that order of priority (they answer the main questions).

In [3]:
import torch, subprocess
print(torch.__version__, 'cuda:', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)

2.11.0+cu128 cuda: True
GPU 0: Tesla T4 (UUID: GPU-16f3c5c0-00ac-dc24-73aa-ae357d171ea9)
GPU 1: Tesla T4 (UUID: GPU-aec0aa28-b86c-f7ae-c95d-ef2d25a748ef)



## 0. Get the code (upload `tree_nn_project_final.zip`, or clone the private repo)

In [2]:
# Add Data -> Upload: a dataset containing tree_nn_project_final.zip (and, optionally, dd_results.json from your first run)
import glob, os, shutil
zp = glob.glob('/kaggle/input/**/tree_nn_project_final*.zip', recursive=True)
print(zp)
%cd /kaggle/working
!unzip -q -o "{zp[0]}" -d /kaggle/working
%cd /kaggle/working/tree_nn_project
!ls tests tree_nn/augment.py

[]
/kaggle/working
unzip:  cannot find or open {zp[0]}, {zp[0]}.zip or {zp[0]}.ZIP.

No zipfiles found.
[Errno 2] No such file or directory: '/kaggle/working/tree_nn_project'
/kaggle/working
ls: cannot access 'tests': No such file or directory
ls: cannot access 'tree_nn/augment.py': No such file or directory


In [6]:
import glob, os, zipfile, shutil
for r, d, f in os.walk('/kaggle/input'):
    print(r, d[:5], f[:5])
    if r.count('/') > 6: break
zp = glob.glob('/kaggle/input/**/tree_nn_project_final*.zip', recursive=True)
print('zip:', zp)
if zp:
    zipfile.ZipFile(zp[0]).extractall('/kaggle/working')
else:
    src = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/tree_nn/augment.py', recursive=True)]
    print('unpacked at:', src)
    assert src, 'neither zip nor folder found'
    shutil.copytree(os.path.dirname(src[0]), '/kaggle/working/tree_nn_project', dirs_exist_ok=True)
%cd /kaggle/working/tree_nn_project
!ls tests tree_nn/augment.py

/kaggle/input ['datasets'] []
/kaggle/input/datasets ['ronakraichandani2108'] []
/kaggle/input/datasets/ronakraichandani2108 ['tree-nn-project-final'] []
/kaggle/input/datasets/ronakraichandani2108/tree-nn-project-final ['tree_nn_project'] []
/kaggle/input/datasets/ronakraichandani2108/tree-nn-project-final/tree_nn_project ['examples', 'notebooks', 'scripts', 'tests', 'tree_nn'] ['.gitignore', 'README.md', 'interpretation.md', 'interpretation_aug_template.md', 'interpretation_template.md']
/kaggle/input/datasets/ronakraichandani2108/tree-nn-project-final/tree_nn_project/examples ['demo_digits', 'pilot_digits', 'pilot_digits_repair'] []
zip: []
unpacked at: ['/kaggle/input/datasets/ronakraichandani2108/tree-nn-project-final/tree_nn_project/tree_nn']
/kaggle/working/tree_nn_project
tree_nn/augment.py

tests:
test_analysis_pipeline.py  test_aug_ensemble.py  test_tree_logic.py


In [ ]:
# ALTERNATIVE: private GitHub repo (token needs 'repo' scope; never share a notebook that contains the token)
# TOKEN='ghp_...'; USER='ronakraichandani'; REPO='MLOAssignment_2026EEY7568_Ronak'
# (not needed on Kaggle)

In [9]:
!pip -q install reportlab scikit-learn matplotlib pillow datasets
RES = '/kaggle/working/results_aug'                         # everything of this notebook goes here
!mkdir -p {RES}

### Fast CIFAR-10 download (Hugging Face). `tree_nn/data.py` uses `data/cifar10.npz` when it exists.

In [10]:
import numpy as np, os
from datasets import load_dataset
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/cifar10.npz'):
    ds = load_dataset('uoft-cs/cifar10')
    def arr(split):
        X = np.stack([np.array(im.convert('RGB')) for im in ds[split]['img']]).astype(np.uint8)
        return X, np.array(ds[split]['label'])
    Xtr, ytr = arr('train'); Xte, yte = arr('test')
    np.savez('data/cifar10.npz', Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte)
print(np.load('data/cifar10.npz')['Xtr'].shape)

README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 23.9MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

(50000, 32, 32, 3)


## 1. Unit tests (CPU, ~1 min). All must print PASSED / OK

In [11]:
!python tests/test_tree_logic.py
!python tests/test_aug_ensemble.py
!python tests/test_analysis_pipeline.py

[paper example] nodes=1 acc=1.0
[multi use_B=False] nodes=538 depth=10 params=8107 train acc by depth=[0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.855, 0.945, 1.0]
   pruned(min_pos=5): params=4642 acc=0.874
[multi use_B=True] nodes=871 depth=10 params=13102 train acc by depth=[0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.857, 0.929, 0.979, 1.0]
   pruned(min_pos=5): params=7477 acc=0.870
[binary] nodes=492 depth=10 params=6405 acc=1.0
ALL TESTS PASSED
ALL TESTS PASSED
M=5: single 0.6516  hard vote(M) 0.9333  soft vote(M) 0.9333  | with TTA: single 0.7850 hard 0.9917  | disagreement 0.558  oracle 0.993
n=  100 A=1 M=4: root: err 0.541 bias2 0.387 var 0.545  tree: err 0.439 bias2 0.298 var 0.464  mlp: err 0.507 bias2 0.364 var 0.513
n=  200 A=1 M=4: root: err 0.528 bias2 0.384 var 0.528  tree: err 0.437 bias2 0.286 var 0.472  mlp: err 0.490 bias2 0.338 var 0.509
n=  100 A=4 M=4: root: err 0.540 bias2 0.391 var 0.540  tree: err 0.447 bias2 0.297 var 0.478  mlp: err 0.

## 2. Quick trial: a tiny version of EVERY script (~10 min). If anything crashes, send me the traceback.
Look at the printed numbers only for plausibility (tiny data), and at the seconds, to scale the long runs.

In [12]:
TRIAL = '/kaggle/working/trial'
!rm -rf {TRIAL}
!python scripts/aug_tree.py --n_train 2000 --A_list 1,2 --widths 64 --max_nodes 200 --steps 300 --trunk_epochs 2 --rhos 0.05 --out {TRIAL}/aug_tree
!python scripts/ensemble.py --n_train 2000 --A 2 --M 2 --widths 64 --max_nodes 200 --steps 300 --trunk_epochs 2 --out {TRIAL}/ens
!python scripts/bias_variance.py --mode trees --sizes 300,600 --M 2 --A_list 1,2 --trunk_n 2000 --trunk_epochs 2 --widths 64 --max_nodes 100 --steps 300 --out {TRIAL}/bvt
!python scripts/bias_variance.py --mode nn --n 500 --M 2 --k_list 4 --epochs 4 --eval_every 2 --out {TRIAL}/bvn
!python scripts/landscape.py --bv_dir {TRIAL}/bvn --n_eval 300 --hess_n 100 --n_dirs 1 --out {TRIAL}/ls
!python scripts/double_descent.py --n_train 1500 --widths 2 --epochs 3 --eval_every 1 --augment --out {TRIAL}/dd

train 2000 test 10000 noisy labels 293
  trunk epoch 1/2  loss 2.208  train acc 0.178
  trunk epoch 2/2  loss 2.022  train acc 0.257
trunk + 2 train views + 5 test views: 10s
A1: nodes 18 params 592765 depth 2 train(all views) 1.0000 | test clean 0.3293 TTA-vote 0.3224 root-only 0.3316 | fresh-aug train acc (clean labels) 0.323 | AUROC {'repair_depth_clean': 0.534619544897441, 'repair_depth_mean': 0.534619544897441, 'root_miss_frac': 0.539401100867538} [5s]
A2: nodes 21 params 691462 depth 2 train(all views) 1.0000 | test clean 0.3187 TTA-vote 0.3560 root-only 0.3436 | fresh-aug train acc (clean labels) 0.387 | AUROC {'repair_depth_clean': 0.6072676051832346, 'repair_depth_mean': 0.644098482258358, 'root_miss_frac': 0.6619810817133226} [7s]
DONE
  trunk epoch 1/2  loss 2.214  train acc 0.171
  trunk epoch 2/2  loss 2.014  train acc 0.268
member 0: nodes 19 params 625664 train(all views) 1.0000 test 0.3140 root-only 0.3285  [feat 7s, tree 6s]
  trunk epoch 1/2  loss 2.213  train acc 0.1

In [16]:
import subprocess, os
os.chdir('/kaggle/working/tree_nn_project')
cmd = ("python scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 "
       "--widths 2,4,8,16,32,64 --augment --out /kaggle/working/results_aug/dd_cifar10_aug "
       "> /kaggle/working/dd.log 2>&1")
subprocess.Popen(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES='1'))

<Popen: returncode: None args: 'python scripts/double_descent.py --dataset c...>

In [17]:
import torch, subprocess
print(torch.cuda.is_available(), torch.cuda.device_count())
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)

True 2
Sun Oct 25 14:12:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------------------

In [18]:
!ps aux | grep -E "[d]ouble_descent|[i]pykernel" | cut -c1-200
!ls -l --time-style=full-iso /kaggle/working/results_aug/dd_cifar10_aug
!date

root         385  0.0  0.0   2812  1968 ?        S    14:12   0:00 /bin/sh -c python scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 --widths 2,4,8,16,32,64 --augment --out /k
root         386  100  4.8 12246404 1585368 ?    Sl   14:12   2:13 python3 scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 --widths 2,4,8,16,32,64 --augment --out /kaggle/work
root         388  0.0  0.0   2812  1844 ?        S    14:12   0:00 /bin/sh -c python scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 --widths 2,4,8,16,32,64 --augment --out /k
root         389 99.9  4.8 12247732 1586348 ?    Rl   14:12   2:13 python3 scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 --widths 2,4,8,16,32,64 --augment --out /kaggle/work
total 0
Sun Oct 25 02:14:34 PM UTC 2026


In [19]:
!pkill -f "[d]ouble_descent.py"
import time; time.sleep(3)
!ps aux | grep -c "[d]ouble_descent"
!rm -rf /kaggle/working/results_aug/dd_cifar10_aug

0


In [20]:
import subprocess, os
os.chdir('/kaggle/working/tree_nn_project')
cmd = ("python scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 "
       "--widths 2,4,8,16,32,64 --augment --out /kaggle/working/results_aug/dd_cifar10_aug "
       "> /kaggle/working/dd.log 2>&1")
subprocess.Popen(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES='1'))

<Popen: returncode: None args: 'python scripts/double_descent.py --dataset c...>

In [23]:
!tail -3 /kaggle/working/dd.log
!date
!ps aux | grep -c "[d]ouble_descent.py"

  k=  8 ep    1  train err 0.7576  test err 0.7247
  k=  8 ep    5  train err 0.6247  test err 0.5832
  k=  8 ep   10  train err 0.5621  test err 0.5100
Sun Oct 25 02:21:33 PM UTC 2026
2


In [24]:
import time, subprocess
for _ in range(60):
    print(subprocess.run("tail -2 /kaggle/working/dd.log; date", shell=True, capture_output=True, text=True).stdout)
    time.sleep(10)

  k= 16 ep   20  train err 0.4033  test err 0.3648
  k= 16 ep   25  train err 0.3950  test err 0.3653
Sun Oct 25 02:24:39 PM UTC 2026

  k= 16 ep   25  train err 0.3950  test err 0.3653
  k= 16 ep   30  train err 0.3633  test err 0.3400
Sun Oct 25 02:24:49 PM UTC 2026

  k= 16 ep   35  train err 0.3401  test err 0.3435
  k= 16 ep   40  train err 0.3198  test err 0.3257
Sun Oct 25 02:24:59 PM UTC 2026

  k= 16 ep   40  train err 0.3198  test err 0.3257
  k= 16 ep   45  train err 0.3191  test err 0.3373
Sun Oct 25 02:25:09 PM UTC 2026

  k= 16 ep   45  train err 0.3191  test err 0.3373
  k= 16 ep   50  train err 0.2973  test err 0.3353
Sun Oct 25 02:25:19 PM UTC 2026

  k= 16 ep   55  train err 0.2803  test err 0.3261
  k= 16 ep   60  train err 0.2743  test err 0.3356
Sun Oct 25 02:25:29 PM UTC 2026

  k= 16 ep   60  train err 0.2743  test err 0.3356
  k= 16 ep   65  train err 0.2463  test err 0.3343
Sun Oct 25 02:25:39 PM UTC 2026

  k= 16 ep   70  train err 0.2349  test err 0.3329
  k=

KeyboardInterrupt: 

**2-HOUR VERSION** (reduced sizes; est. ~2 h on a T4, only the double-descent timing is measured). Read the seconds printed by the quick trial and each cell; if a cell runs long, stop it, results are saved per item and resume on re-run. Priority if time runs out: sections 4 and 5, then 3, then 6 and 7.

## 3. Double descent WITH augmentation (random crop + flip, fresh every epoch)
Same recipe as your first run (15k images, 15% label noise, Adam 1e-4, 120 epochs), so the curves are directly comparable.
The no-augmentation baseline is copied from your earlier results. Cheaper: drop widths 48 and 64 (they take ~7 and ~10 min).

In [40]:
!tail -2 /kaggle/working/dd.log
!ls /kaggle/working/results_aug/dd_cifar10 /kaggle/working/results_aug/dd_cifar10_aug

  k= 64 ep   10  train err 0.3446  test err 0.3034
  k= 64 ep   15  train err 0.3107  test err 0.2863
/kaggle/working/results_aug/dd_cifar10:
dd_results.json

/kaggle/working/results_aug/dd_cifar10_aug:
dd_results.json  double_descent.png


In [14]:
import glob, os, shutil
os.makedirs(RES + '/dd_cifar10', exist_ok=True)
b = glob.glob('/kaggle/input/**/dd_results.json', recursive=True)      # no-augmentation baseline from your first run
print('baseline:', b)
if b: shutil.copy(b[0], RES + '/dd_cifar10/dd_results.json')
!python scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 --widths 2,4,8,16,32,64 --augment --out {RES}/dd_cifar10_aug

baseline: ['/kaggle/input/datasets/ronakraichandani2108/dd-results/dd_results.json']
  k=  2 ep    1  train err 0.9016  test err 0.8988
  k=  2 ep    5  train err 0.8296  test err 0.8159
  k=  2 ep   10  train err 0.7489  test err 0.7244
  k=  2 ep   15  train err 0.7179  test err 0.6774
  k=  2 ep   20  train err 0.6948  test err 0.6580
  k=  2 ep   25  train err 0.6779  test err 0.6410
  k=  2 ep   30  train err 0.6676  test err 0.6287
  k=  2 ep   35  train err 0.6611  test err 0.6144
  k=  2 ep   40  train err 0.6537  test err 0.6092
  k=  2 ep   45  train err 0.6429  test err 0.5949
  k=  2 ep   50  train err 0.6341  test err 0.5911
  k=  2 ep   55  train err 0.6343  test err 0.5847
  k=  2 ep   60  train err 0.6266  test err 0.5811
  k=  2 ep   65  train err 0.6197  test err 0.5733
  k=  2 ep   70  train err 0.6177  test err 0.5699
  k=  2 ep   75  train err 0.6166  test err 0.5664
  k=  2 ep   80  train err 0.6092  test err 0.5619
  k=  2 ep   85  train err 0.6071  test err 0.55

In [22]:
!tail -4 /kaggle/working/dd.log
!date
!ls /kaggle/working/results_aug/dd_cifar10_aug

  k=  4 ep   10  train err 0.6559  test err 0.6135
  k=  4 ep   15  train err 0.6203  test err 0.5711
  k=  4 ep   20  train err 0.5997  test err 0.5456
  k=  4 ep   25  train err 0.5846  test err 0.5305
Sun Oct 25 02:19:17 PM UTC 2026
dd_results.json  double_descent.png


## 4. One tree grown on A augmented views (the tree must interpolate all A x N points)
`--noise 0.15`, 20,000 images, A in {1, 2, 4}. Check that every row says `train(all views) 1.0000`; a printed WARNING means the
depth/node cap stopped the tree and its numbers are not comparable. The second command is the tree-wise sweep (fixed node width W)
with and without augmentation.

In [ ]:
!python scripts/aug_tree.py --dataset cifar10 --noise 0.15 --n_train 10000 --A_list 1,2,4 --out {RES}/aug_tree_cifar10

In [25]:
import subprocess, os
os.chdir('/kaggle/working/tree_nn_project')
cmd = ("python scripts/aug_tree.py --dataset cifar10 --noise 0.15 --n_train 10000 --A_list 1,2,4 "
       "--out /kaggle/working/results_aug/aug_tree_cifar10 > /kaggle/working/aug_tree.log 2>&1")
subprocess.Popen(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'))

<Popen: returncode: None args: 'python scripts/aug_tree.py --dataset cifar10...>

In [26]:
import subprocess, os
os.chdir('/kaggle/working/tree_nn_project')
cmd = ("python scripts/ensemble.py --dataset cifar10 --noise 0.15 --n_train 10000 --A 4 --M 5 "
       "--diversity both --out /kaggle/working/results_aug/ens_cifar10 > /kaggle/working/ens.log 2>&1")
subprocess.Popen(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES='1'))

<Popen: returncode: None args: 'python scripts/ensemble.py --dataset cifar10...>

In [28]:
!tail -4 /kaggle/working/aug_tree.log
!tail -4 /kaggle/working/ens.log

  trunk epoch 6/6  loss 1.679  train acc 0.432
trunk + 4 train views + 5 test views: 16s
A1: nodes 1 params 133898 depth 0 train(all views) 1.0000 | test clean 0.4407 TTA-vote 0.4665 root-only 0.4407 | fresh-aug train acc (clean labels) 0.446 | AUROC {'repair_depth_clean': 0.5, 'repair_depth_mean': 0.5, 'root_miss_frac': 0.5} [5s]
A2: nodes 11 params 561576 depth 1 train(all views) 1.0000 | test clean 0.4517 TTA-vote 0.5113 root-only 0.4578 | fresh-aug train acc (clean labels) 0.527 | AUROC {'repair_depth_clean': 0.5312254154748053, 'repair_depth_mean': 0.5455579367333357, 'root_miss_frac': 0.5520417718562327} [40s]
  trunk epoch 3/6  loss 1.784  train acc 0.380
  trunk epoch 4/6  loss 1.730  train acc 0.406
  trunk epoch 5/6  loss 1.686  train acc 0.434
  trunk epoch 6/6  loss 1.628  train acc 0.456


In [29]:
!tail -3 /kaggle/working/aug_tree.log
!tail -3 /kaggle/working/ens.log

trunk + 4 train views + 5 test views: 16s
A1: nodes 1 params 133898 depth 0 train(all views) 1.0000 | test clean 0.4407 TTA-vote 0.4665 root-only 0.4407 | fresh-aug train acc (clean labels) 0.446 | AUROC {'repair_depth_clean': 0.5, 'repair_depth_mean': 0.5, 'root_miss_frac': 0.5} [5s]
A2: nodes 11 params 561576 depth 1 train(all views) 1.0000 | test clean 0.4517 TTA-vote 0.5113 root-only 0.4578 | fresh-aug train acc (clean labels) 0.527 | AUROC {'repair_depth_clean': 0.5312254154748053, 'repair_depth_mean': 0.5455579367333357, 'root_miss_frac': 0.5520417718562327} [40s]
  trunk epoch 4/6  loss 1.730  train acc 0.406
  trunk epoch 5/6  loss 1.686  train acc 0.434
  trunk epoch 6/6  loss 1.628  train acc 0.456


In [30]:
!ps aux | grep -E "[a]ug_tree|[e]nsemble.py|[d]ouble_descent" | cut -c1-110
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv
!date

root         600  0.0  0.0   2812  1980 ?        S    14:35   0:00 /bin/sh -c python scripts/aug_tree.py --dat
root         601  101  6.1 13144248 2007868 ?    Rl   14:35   3:27 python3 scripts/aug_tree.py --dataset cifar
root         614  0.0  0.0   2812  2012 ?        S    14:35   0:00 /bin/sh -c python scripts/ensemble.py --dat
root         615  100  5.5 12560104 1823708 ?    Rl   14:35   2:58 python3 scripts/ensemble.py --dataset cifar
index, utilization.gpu [%], memory.used [MiB]
0, 100 %, 923 MiB
1, 100 %, 923 MiB
Sun Oct 25 02:38:37 PM UTC 2026


In [31]:
!tail -4 /kaggle/working/dd.log
!python -c "import json;d=json.load(open('/kaggle/working/results_aug/dd_cifar10_aug/dd_results.json'));print(sorted(d['runs'].keys(), key=int))"

  File "/usr/local/lib/python3.13/dist-packages/torch/amp/grad_scaler.py", line 371, in <genexpr>
    if not sum(v.item() for v in optimizer_state["found_inf_per_device"].values()):
               ~~~~~~^^
KeyboardInterrupt
['2', '4', '8', '16', '32']


In [32]:
import subprocess, os
os.chdir('/kaggle/working/tree_nn_project')
cmd = ("python scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 "
       "--widths 2,4,8,16,32,64 --augment --out /kaggle/working/results_aug/dd_cifar10_aug "
       ">> /kaggle/working/dd.log 2>&1")
subprocess.Popen(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES='1'))

<Popen: returncode: None args: 'python scripts/double_descent.py --dataset c...>

In [51]:
!tail -2 /kaggle/working/dd.log

  k= 64 ep   45  train err 0.0962  test err 0.2964
  k= 64 ep   50  train err 0.0777  test err 0.3145


In [53]:
!tail -3 /kaggle/working/aug_tree.log
!tail -3 /kaggle/working/ens.log

A2: nodes 11 params 561576 depth 1 train(all views) 1.0000 | test clean 0.4517 TTA-vote 0.5113 root-only 0.4578 | fresh-aug train acc (clean labels) 0.527 | AUROC {'repair_depth_clean': 0.5312254154748053, 'repair_depth_mean': 0.5455579367333357, 'root_miss_frac': 0.5520417718562327} [40s]
A4: nodes 18 params 1680061 depth 2 train(all views) 1.0000 | test clean 0.4581 TTA-vote 0.5144 root-only 0.4676 | fresh-aug train acc (clean labels) 0.569 | AUROC {'repair_depth_clean': 0.5834412145463104, 'repair_depth_mean': 0.6532537294873983, 'root_miss_frac': 0.6791368422387895} [191s]
DONE
  trunk epoch 4/6  loss 1.734  train acc 0.406
  trunk epoch 5/6  loss 1.703  train acc 0.423
  trunk epoch 6/6  loss 1.645  train acc 0.455


## 5. Ensembles of trees with majority voting
Main run: 8 trees, A=4 views each, diversity = bootstrap + separately trained trunk. Then two ablations: seed/augmentation-only diversity and no
augmentation (A=1). Each member is saved as soon as it is done; re-run the cell after a disconnect. Cheaper: `--M 5`.

In [ ]:
!python scripts/ensemble.py --dataset cifar10 --noise 0.15 --n_train 10000 --A 4 --M 5 --diversity both --out {RES}/ens_cifar10

In [ ]:
# ablations (optional, run after the main ensemble)
# skipped in the 2-hour version (uncomment if time is left)
#!python scripts/ensemble.py --dataset cifar10 --noise 0.15 --n_train 10000 --A 1 --M 5 --diversity both --out {RES}/ens_cifar10_noaug

## 6. Bias and variance
(a) vs number of training samples n: interpolating tree vs its root node vs one wide MLP, A in {1, 4}, 5 disjoint training sets per n.
(b) vs epochs: ResNet-18(k) on 4 disjoint 5,000-image noisy subsets, with and without augmentation.

In [ ]:
!python scripts/bias_variance.py --mode trees --dataset cifar10 --noise 0.15 --sizes 1000,2000,4000 --M 4 --A_list 1,4 --out {RES}/bv_trees

In [ ]:
!python scripts/bias_variance.py --mode nn --dataset cifar10 --img_size 32 --noise 0.15 --n 5000 --M 3 --k_list 4,16 --epochs 60 --out {RES}/bv_nn

In [44]:
import subprocess, os
os.chdir('/kaggle/working/tree_nn_project')
R = '/kaggle/working/results_aug'
cmd = (f"python scripts/bias_variance.py --mode trees --dataset cifar10 --noise 0.15 --sizes 1000,2000,4000 --M 4 --A_list 1,4 --out {R}/bv_trees > /kaggle/working/bvt.log 2>&1; "
       f"python scripts/bias_variance.py --mode nn --dataset cifar10 --img_size 32 --noise 0.15 --n 5000 --M 3 --k_list 4,16 --epochs 60 --out {R}/bv_nn > /kaggle/working/bvn.log 2>&1; "
       f"python scripts/landscape.py --bv_dir {R}/bv_nn --dataset cifar10 --img_size 32 --noise 0.15 --hess_n 128 --n_dirs 2 --out {R}/landscape > /kaggle/working/ls.log 2>&1")
subprocess.Popen(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'))

<Popen: returncode: None args: 'python scripts/bias_variance.py --mode trees...>

In [55]:
!tail -4 /kaggle/working/bvt.log
!grep member /kaggle/working/ens.log

n= 1000 A=4 M=4: root: err 0.592 bias2 0.542 var 0.470  tree: err 0.592 bias2 0.542 var 0.470  mlp: err 0.577 bias2 0.544 var 0.411
n= 2000 A=4 M=4: root: err 0.596 bias2 0.541 var 0.506  tree: err 0.596 bias2 0.541 var 0.506  mlp: err 0.569 bias2 0.532 var 0.421
n= 4000 A=4 M=4: root: err 0.569 bias2 0.520 var 0.454  tree: err 0.569 bias2 0.520 var 0.454  mlp: err 0.567 bias2 0.516 var 0.456
DONE
member 0: nodes 15 params 1480948 train(all views) 1.0000 test 0.4591 root-only 0.4780  [feat 17s, tree 215s]
member 1: nodes 11 params 857640 train(all views) 1.0000 test 0.4523 root-only 0.4554  [feat 16s, tree 226s]
member 2: nodes 11 params 561576 train(all views) 1.0000 test 0.4603 root-only 0.4624  [feat 31s, tree 149s]


In [54]:
import subprocess, os
os.chdir('/kaggle/working/tree_nn_project')
R = '/kaggle/working/results_aug'
cmd = (f"python scripts/ensemble.py --dataset cifar10 --noise 0.15 --n_train 10000 --A 1 --M 5 --diversity both --out {R}/ens_cifar10_noaug > /kaggle/working/ens_noaug.log 2>&1; "
       f"python scripts/ensemble.py --dataset cifar10 --noise 0.15 --n_train 10000 --A 4 --M 5 --diversity seed --out {R}/ens_cifar10_seed > /kaggle/working/ens_seed.log 2>&1")
subprocess.Popen(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES='1'))

<Popen: returncode: None args: 'python scripts/ensemble.py --dataset cifar10...>

In [60]:
!tail -3 /kaggle/working/bvn.log
!tail -2 /kaggle/working/ens_noaug.log
!tail -2 /kaggle/working/dd.log

k4_aug0: M=3  best epoch 15 (test err 0.700, bias2 0.781, var 0.031) -> end: test err 0.742, bias2 0.733, var 0.182
k4_aug1: M=3  best epoch 60 (test err 0.565, bias2 0.660, var 0.041) -> end: test err 0.565, bias2 0.660, var 0.041
DONE
M=5: single 0.4574  hard vote(M) 0.5641  soft vote(M) 0.5676  | with TTA: single 0.4724 hard 0.5646  | disagreement 0.570  oracle 0.797
DONE
  k= 64 ep  120  train err 0.0297  test err 0.3098
DONE


In [63]:
!tail -4 /kaggle/working/ens.log
!tail -3 /kaggle/working/ls.log
!tail -2 /kaggle/working/ens_seed.log

  trunk epoch 6/6  loss 1.643  train acc 0.450
member 4: nodes 12 params 989227 train(all views) 1.0000 test 0.4556 root-only 0.4609  [feat 44s, tree 372s]
M=5: single 0.4562  hard vote(M) 0.5601  soft vote(M) 0.5632  | with TTA: single 0.5061 hard 0.5909  | disagreement 0.577  oracle 0.807
DONE
k4_aug0: params 44622  train loss 0.7445  test err 0.727  top Hessian eig 4352.6  d(loss) at rho=0.05: 0.490  barrier(test loss) 0.29
k4_aug1: params 44622  train loss 1.7406  test err 0.562  top Hessian eig 1187.1  d(loss) at rho=0.05: 0.051  barrier(test loss) 0.68
DONE
member 0: nodes 17 params 1647162 train(all views) 1.0000 test 0.4560 root-only 0.4758  [feat 50s, tree 485s]
member 1: nodes 19 params 1712960 train(all views) 1.0000 test 0.4645 root-only 0.4857  [feat 13s, tree 225s]


## 7. Loss landscape of the saved ResNets (slices, sharpness, Hessian eigenvalue, linear paths)

In [ ]:
!python scripts/landscape.py --bv_dir {RES}/bv_nn --dataset cifar10 --img_size 32 --noise 0.15 --hess_n 128 --n_dirs 2 --out {RES}/landscape

## 8. Look at the results, send them back, build the report
`results_aug_light.zip` contains only JSON and PNG files (small). Upload it to the chat so the interpretation can be written from your real numbers.
Then (after `interpretation_aug.md` exists in the project folder) build the brief report.

In [71]:
import zipfile, os
with zipfile.ZipFile('/kaggle/working/results_aug_light.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for r, _, fs in os.walk(RES):
        for f in fs:
            if f.endswith(('.json', '.png')): z.write(os.path.join(r, f), os.path.relpath(os.path.join(r, f), RES))
print(os.path.getsize('/kaggle/working/results_aug_light.zip'), 'bytes -> download from the Output tab (right panel)')

1139340 bytes -> download from the Output tab (right panel)


In [64]:
!ls /kaggle/working/results_aug/landscape

landscape.json	landscape.png


In [65]:
!tail -3 /kaggle/working/ens_seed.log

member 1: nodes 19 params 1712960 train(all views) 1.0000 test 0.4645 root-only 0.4857  [feat 13s, tree 225s]
member 2: nodes 17 params 1647162 train(all views) 1.0000 test 0.4574 root-only 0.4734  [feat 8s, tree 220s]
member 3: nodes 17 params 1548474 train(all views) 1.0000 test 0.4575 root-only 0.4732  [feat 8s, tree 214s]


In [69]:
!tail -3 /kaggle/working/ens_seed.log

member 4: nodes 21 params 1778758 train(all views) 1.0000 test 0.4519 root-only 0.4604  [feat 8s, tree 234s]
M=5: single 0.4574  hard vote(M) 0.5189  soft vote(M) 0.5179  | with TTA: single 0.5174 hard 0.5631  | disagreement 0.481  oracle 0.742
DONE


In [ ]:
REPO_URL = 'https://github.com/ronakraichandani/MLOAssignment_2026EEY7568_Ronak'   # check
!python make_report_aug.py --results {RES} --repo_url {REPO_URL} --interpretation interpretation_aug.md --out report_aug.pdf
print('report_aug.pdf is in /kaggle/working (Output tab)')